In [1]:
!git clone -b continued-pretraining https://github.com/namantuli18/arc-prize-2024.git


Cloning into 'arc-prize-2024'...
remote: Enumerating objects: 818, done.
remote: Counting objects: 100% (222/222), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 818 (delta 218), reused 191 (delta 191), pack-reused 596 (from 3)
Receiving objects: 100% (818/818), 2.86 MiB | 26.61 MiB/s, done.
Resolving deltas: 100% (565/565), done.


In [2]:
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip uninstall transformers -y && pip install --upgrade --no-cache-dir "git+https://github.com/huggingface/transformers.git"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.0/310.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.3/208.3 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 20.9 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
Found existing installation: unsloth 2025.9.5
Uninstalling unsloth-2025.9.5:
  Successfully uninstalled unsloth-202

In [3]:
cd /content/arc-prize-2024/training_code


/content/arc-prize-2024/training_code


In [4]:
!pip install diskcache

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.8 MB/s eta 0:00:00


In [5]:
# !pip install transformers==4.44.0
!pip install trl==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 335.7/335.7 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: trl
    Found existing installation: trl 0.23.0
    Uninstalling trl-0.23.0:
      Successfully uninstalled trl-0.23.0


In [ ]:
# !pip install unsloth_zoo

In [6]:
cd /content/arc-prize-2024/training_code

/content/arc-prize-2024/training_code


In [8]:
# Copyright 2024 Daniel Franzen and Jan Disselhoff
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
import unsloth
import os
import json
from unsloth import FastLanguageModel
from unsloth import UnslothTrainer as Trainer, unsloth_train, is_bfloat16_supported
from unsloth import UnslothTrainingArguments as TrainingArguments
from datasets import Dataset
from diskcache import Cache

from arc_loader import ArcDataset
from model_tools import InputMaskingDataCollator
from model_tools import load_unsloth_4bit, save_model_and_tokenizer
from inference_tools import inference_run
from selection import EvalTool
from arc_downloader import download_arc_data

# input paths
base_model = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'  # auto-downloaded from huggingface.co
arc_data_path = '/content/ARC'  # as on kaggle arc prize 2024
download_arc_data(arc_data_path)

# output paths
output_path = 'output_evaluation_Llama-rearc_with_ttt'
save_model_path = os.path.join(output_path, 'finetuned_model')
inference_cache = os.path.join(output_path, 'inference_cache')
submission_file = os.path.join(output_path, 'submission.json')

# load evaluation dataset
arc_eval_set = ArcDataset.load_from_json(os.path.join(arc_data_path, 'arc-agi_evaluation_challenges.json'))
arc_eval_set = arc_eval_set.load_solutions(os.path.join(arc_data_path, 'arc-agi_evaluation_solutions.json'))

# load model
retrain = not os.path.exists(save_model_path)
model, tokenizer = load_unsloth_4bit(base_model if retrain else save_model_path)

# set formatting options
fmt_opts = dict(
    preprompt='ABCDEFGHJKLMNPQRSTUVWXYZabcdefghjklmnpqrstuvwxyz',
    query_beg='I',
    reply_beg='\n+/-=O',
    reply_end='\n' + tokenizer.eos_token,
    lines_sep='\n',
    max_tokens=128000,
)

if retrain:
    # create lora model
    model = FastLanguageModel.get_peft_model(
        model=model,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj',
                        'embed_tokens', 'lm_head'],
        r=64,
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing=True,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    # augment data set and transform to list (eventually removing examples to stay below the max. token count)
    train_aug_opts = dict(tp=True, rt=True, perm=True, shfl_ex=True, seed=0)
    train_dataset_augment = arc_eval_set.remove_test_data().repeat(n=48, seed=0).augment(**train_aug_opts)
    train_dataset_as_list = train_dataset_augment.as_list(len_name='text', **fmt_opts)

    MAX_LEN = 2048

    tokenized_dataset = tokenizer(
        [x['text'] for x in train_dataset_as_list],
        return_tensors=None,
        padding=False,
        truncation=True,
        max_length=MAX_LEN,
    )
    tokenized_dataset["labels"] = tokenized_dataset["input_ids"].copy()

    from datasets import Dataset
    train_dataset = Dataset.from_dict(tokenized_dataset)


    # run test-time training
    FastLanguageModel.for_training(model)
    trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        dataset_text_field="text",
        max_seq_length=fmt_opts['max_tokens'],
        data_collator=InputMaskingDataCollator(
            instruction_template=fmt_opts['query_beg'],
            response_template=fmt_opts['reply_beg'],
            mlm=False,
            tokenizer=tokenizer,
            mask_first_n_examples=0,
        ),
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=2,
            warmup_ratio=0.25,
            num_train_epochs=1,
            learning_rate=1e-4,
            embedding_learning_rate=1e-5,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            logging_steps=10,
            optim="adamw_8bit",
            weight_decay=0.00,
            lr_scheduler_type='cosine',
            seed=42,
            output_dir='tmp_output',
            save_strategy='no',
            report_to='none',
        ),
    )
    trainer_stats = unsloth_train(trainer)
    save_model_and_tokenizer(save_model_path, model, tokenizer)

# run inference
FastLanguageModel.for_inference(model)
infer_aug_opts = dict(tp='all', rt='all', perm=True, shfl_ex=True, seed=10000)
infer_dataset = arc_eval_set.repeat(2).augment(**infer_aug_opts)
model_cache = Cache(inference_cache).memoize(typed=True, ignore=set(['model_tok', 'guess']))
eval_tool = EvalTool(n_guesses=2)
inference_results = inference_run(
    model_tok=(model, tokenizer),
    fmt_opts=fmt_opts,
    dataset=infer_dataset,
    min_prob=0.1,
    aug_score_opts=infer_aug_opts,
    callback=eval_tool.process_result,
    cache=model_cache,
)

# write submission
with open(submission_file, 'w') as f:
    json.dump(arc_eval_set.get_submission(inference_results), f)
with open(submission_file, 'r') as f:
    print(f"Score for '{submission_file}':", arc_eval_set.validate_submission(json.load(f)))

==((====))==  Unsloth 2025.9.5: Fast Qwen2 patching. Transformers: 4.57.0.dev0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM


convert dataset: 100%|██████████| 19200/19200 [00:13<00:00, 1464.83it/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 19,200 | Num Epochs = 1 | Total steps = 4,800
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 742,064,128 of 3,828,002,816 (19.39% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,0.516800
20,0.590200
30,0.506400
40,0.519600
50,0.554000
60,0.414800
70,0.370700
80,0.421000
90,0.391100
100,0.398100


KeyboardInterrupt: 

In [ ]:
cd /content/arc-prize-2024/training_code

In [ ]:
# import os

# # Set your W&B API key from Kaggle secrets or environment
# os.environ["WANDB_API_KEY"] = "7a322f7176c44d63b0212a1738308082aaf05000"
# os.environ["WANDB_PROJECT"] = "arc-prize-2025"
# os.environ["WANDB_NAME"] = "mistral-lora-2025-finetune-merge-2epochs"
# os.environ["WANDB_LOG_MODEL"] = "false"

In [ ]:
# !zip -r /content/weights_2epochs_raw-data.zip /content/arc-prize-2024/training_code/pretrained_models
